In [2]:
# !pip install azure-ai-projects azure-core azure-storage-blob

In [4]:
# !pip install --upgrade openai azure-ai-projects azure-identity

In [42]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load env variables
load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

api_key = os.getenv("Analyzer_api")
endpoint = os.getenv("Analyzer_endpoint")

client = OpenAI(
    base_url=f"{endpoint}",
    api_key=api_key
)


In [43]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text", 
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": ["Project Management", "PR", "Leadership"]\n}', type='output_text', logprobs=[])]


In [44]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [45]:
from azure.storage.blob import BlobServiceClient
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

AZURE_STORAGE_CONNECTION_STRING = os.getenv("ConnectionString")

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "naina"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [46]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [47]:
blob_client = blob_service_client.get_blob_client(
    container="naina",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F6C6D199974"',
 'last_modified': datetime.datetime(2026, 4, 21, 6, 8, 36, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xdb\xe3g\xe1$0\xd2G\xd7k\xff\x97\xbd\xe3\xdb`'),
 'client_request_id': '8900ee83-3d48-11f1-b377-844dd23af55c',
 'request_id': '6f511577-801e-0020-4055-d1899f000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 6, 8, 36, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [48]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_113836.txt


In [49]:
# Build & Deploy an AI Research Paper Summarizer (Azure AI Foundry)
# 🎯 Objective
# Students will:

# Create AI resource

# Build an AI agent

# Deploy it

# Call it using Python (Colab)

# Store summarized output in cloud

In [50]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load env variables
load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

api_key = os.getenv("Analyzer_api")
endpoint = os.getenv("Analyzer_endpoint")

client = OpenAI(
    base_url=f"{endpoint}",
    api_key=api_key
)

In [53]:
paper_text = """
Title: AI in Healthcare

This paper discusses the use of AI in medical diagnosis,
improving accuracy and reducing human error...
"""

try:
    response = client.responses.create(
        model="gpt-4.1",   # ⚠️ use deployment name (Azure)
        input=f"""
Summarize the following research paper into JSON format with:

- Title
- Abstract
- Key Contributions (3 points)
- Methodology
- Conclusion
- Limitations

Return ONLY valid JSON.

Paper:
{paper_text}
"""
    )

    print("--- SUMMARY ---")
    print(response.output_text)

except Exception as e:
    print(f"Error: {e}")


--- SUMMARY ---
{
  "Title": "AI in Healthcare",
  "Abstract": "This paper discusses the use of AI in medical diagnosis, improving accuracy and reducing human error. It explores various machine learning models applied to medical imaging, patient data analysis, and predictive analytics. The paper reviews current implementations, their impact on clinical workflows, and identifies key challenges in privacy and data security.",
  "Key Contributions": [
    "Comprehensive review of AI applications in medical diagnosis.",
    "Evaluation of AI's effectiveness in reducing diagnostic errors and improving workflow.",
    "Identification of primary challenges related to data privacy and security in healthcare AI."
  ],
  "Methodology": "The paper utilizes a literature review approach, analyzing recent studies and published data on AI models in healthcare. Comparative analysis of different machine learning techniques is performed with respect to diagnostic accuracy, speed, and integration into cl

In [54]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Title": "AI in Healthcare",
  "Abstract": "This paper discusses the use of AI in medical diagnosis, improving accuracy and reducing human error. It explores various machine learning models applied to medical imaging, patient data analysis, and predictive analytics. The paper reviews current implementations, their impact on clinical workflows, and identifies key challenges in privacy and data security.",
  "Key Contributions": [
    "Comprehensive review of AI applications in medical diagnosis.",
    "Evaluation of AI's effectiveness in reducing diagnostic errors and improving workflow.",
    "Identification of primary challenges related to data privacy and security in healthcare AI."
  ],
  "Methodology": "The paper utilizes a literature review approach, analyzing recent studies and published data on AI models in healthcare. Comparative analysis of different machine learning techniques is performed with respect to diagnostic accuracy, speed, and integr

In [55]:
from azure.storage.blob import BlobServiceClient
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

AZURE_STORAGE_CONNECTION_STRING = os.getenv("ConnectionString")

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "naina"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [56]:
import datetime

file_name = f"paper_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [57]:
blob_client = blob_service_client.get_blob_client(
    container="naina",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F6D3D0EBCC8"',
 'last_modified': datetime.datetime(2026, 4, 21, 6, 14, 25, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\x8fR\x18\xbb\xfa&}\xbb\xf45w\xd0\xea\xd3\xd3\xe7'),
 'client_request_id': '58f6f943-3d49-11f1-8c14-844dd23af55c',
 'request_id': 'd43aad6c-301e-0078-1d56-d151c0000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 6, 14, 25, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [58]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: paper_20260421_114412.txt


In [59]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load env variables
load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

api_key = os.getenv("Analyzer_api")
endpoint = os.getenv("Analyzer_endpoint")

client = OpenAI(
    base_url=f"{endpoint}",
    api_key=api_key
)

In [61]:
report_text = """
Patient: John Doe
Age: 45

Hemoglobin: 10 g/dL (Low)
WBC Count: 12000 (High)

Observation: Possible infection and anemia.
"""

try:
    response = client.responses.create(
        model="gpt-4.1",   
        input=f"""
Interpret the following medical report into JSON format with:

- Patient Info
- Key Findings
- Abnormal Values
- Explanation (in simple terms)
- Disclaimer

Rules:
- Do NOT provide diagnosis
- Do NOT suggest treatment or medicines
- Keep explanation simple

Return ONLY valid JSON.

Report:
{report_text}
"""
    )

    print("--- INTERPRETATION ---")
    print(response.output_text)

except Exception as e:
    print(f"Error: {e}")

--- INTERPRETATION ---
{
  "Patient Info": {
    "Name": "John Doe",
    "Age": 45
  },
  "Key Findings": [
    "Low hemoglobin",
    "High WBC count"
  ],
  "Abnormal Values": {
    "Hemoglobin": "10 g/dL (Low)",
    "WBC Count": "12000 (High)"
  },
  "Explanation": "The report shows that the red blood cell pigment (hemoglobin) is lower than normal, which can lead to feeling tired or weak. The white blood cell count is higher than normal, which usually happens when the body is fighting an infection or is under stress.",
  "Disclaimer": "This report alone cannot be used to make a diagnosis. Please consult a qualified healthcare professional for further evaluation and interpretation."
}


In [62]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Patient Info": {
    "Name": "John Doe",
    "Age": 45
  },
  "Key Findings": [
    "Low hemoglobin",
    "High WBC count"
  ],
  "Abnormal Values": {
    "Hemoglobin": "10 g/dL (Low)",
    "WBC Count": "12000 (High)"
  },
  "Explanation": "The report shows that the red blood cell pigment (hemoglobin) is lower than normal, which can lead to feeling tired or weak. The white blood cell count is higher than normal, which usually happens when the body is fighting an infection or is under stress.",
  "Disclaimer": "This report alone cannot be used to make a diagnosis. Please consult a qualified healthcare professional for further evaluation and interpretation."
}


In [63]:
from azure.storage.blob import BlobServiceClient
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

AZURE_STORAGE_CONNECTION_STRING = os.getenv("ConnectionString")

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "naina"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [64]:
import datetime

file_name = f"report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [65]:
blob_client = blob_service_client.get_blob_client(
    container="naina",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F7481AF484C"',
 'last_modified': datetime.datetime(2026, 4, 21, 7, 6, 27, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'SC\x98\x9e?\xfa\x04j\x13\x11\xfe\xde\xab\xdb\x00\xd1'),
 'client_request_id': '9d969898-3d50-11f1-b608-844dd23af55c',
 'request_id': 'abcb1f30-501e-000c-435d-d16530000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 7, 6, 26, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [66]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: report_20260421_123615.txt
